In [ ]:
# ===========================================
# ML2 — Data Loader & Feature Builder (Colab)
# Builds: baseline_like.csv, features_stats_only.csv, features_odds_aug.csv
# Leakage-safe rolling stats; normalized & aggregated odds
# ===========================================

import os
import re
import glob
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict
from google.colab import drive
drive.mount("/content/gdrive")

# ---------------------------
# PATHS (edit if needed)
# ---------------------------
basedir_data = '/content/gdrive/MyDrive/ML2_SHARED/semestral_project/data'  # your shared data root
outdir = '/content/gdrive/MyDrive/ML2_SHARED/semestral_project/processed'
os.makedirs(outdir, exist_ok=True)

# ---------------------------
# 1) LOAD ALL CSVs
#    Expect structure: {country}/{league}/{season}.csv
# ---------------------------
def load_all_csvs(root: str) -> pd.DataFrame:
    pattern = os.path.join(root, '*', '*', '*.csv')  # country/league/season.csv
    files = sorted(glob.glob(pattern))
    frames = []
    for fp in files:
        try:
            # infer country & league from path
            # .../<country>/<league>/<season>.csv
            parts = fp.split(os.sep)
            country, league = parts[-3], parts[-2]
            df = pd.read_csv(fp)
            df['country'] = country
            df['league'] = league
            df['source_file'] = os.path.basename(fp)
            frames.append(df)
        except Exception as e:
            print(f"[WARN] Skipping {fp}: {e}")
    if not frames:
        raise RuntimeError("No CSV files found under: " + pattern)
    df_all = pd.concat(frames, ignore_index=True, sort=False)
    return df_all

raw = load_all_csvs(basedir_data)
print(f"Loaded rows: {len(raw):,}   columns: {len(raw.columns)}")


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Loaded rows: 42,593   columns: 140


In [ ]:
# ---------------------------
# 2) BASIC CLEANUP & TARGET
# ---------------------------
def parse_and_sort(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Date -> datetime (dd/mm/yy typical in Football-Data)
    if 'Date' in df.columns and not np.issubdtype(df['Date'].dtype, np.datetime64):
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

    # Time -> combine with Date when available
    if 'Time' in df.columns:
        # Best effort parse HH:MM, coercing weird entries to 00:00
        tt = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.strftime('%H:%M')
        tt = tt.fillna('00:00')
        df['kickoff_dt'] = pd.to_datetime(df['Date'].dt.strftime('%Y-%m-%d') + ' ' + tt, errors='coerce')
    else:
        df['kickoff_dt'] = pd.to_datetime(df['Date'], errors='coerce')

    # Drop rows we can't place in time
    df = df[~df['kickoff_dt'].isna()].copy()

    # Season key (approx: season starts in July)
    df['season'] = df['Date'].dt.year.where(df['Date'].dt.month >= 7, df['Date'].dt.year - 1)
    df['month'] = df['Date'].dt.month
    df['weekday'] = df['Date'].dt.weekday
    df['is_weekend'] = (df['weekday'] >= 5).astype(int)

    # Division field guard
    if 'Div' not in df.columns:
        # build Div from country/league if absent
        df['Div'] = df['country'].astype(str) + '_' + df['league'].astype(str)

    # Sort for time-aware processing
    df = df.sort_values(['Div', 'kickoff_dt', 'HomeTeam', 'AwayTeam']).reset_index(drop=True)
    return df

def add_over25_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Ensure goals exist
    if not {'FTHG','FTAG'}.issubset(df.columns):
        raise ValueError("FTHG and FTAG are required to derive the Over/Under target.")
    df['total_goals'] = df['FTHG'].astype(float) + df['FTAG'].astype(float)
    df['y_over25'] = (df['total_goals'] >= 3).astype(int)
    return df

df = parse_and_sort(raw)
df = add_over25_target(df)


In [ ]:
print(df.info())
print(df.head())
print(list(df.keys()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42593 entries, 0 to 42592
Columns: 147 entries, Div to y_over25
dtypes: datetime64[ns](2), float64(125), int32(3), int64(4), object(13)
memory usage: 47.3+ MB
None
  Div       Date   Time          HomeTeam     AwayTeam  FTHG  FTAG FTR  HTHG  \
0  B1 2019-07-26  19:30              Genk     Kortrijk     2     1   H   0.0   
1  B1 2019-07-27  17:00     Cercle Brugge     Standard     0     2   A   0.0   
2  B1 2019-07-27  19:00        St Truiden     Mouscron     0     1   A   0.0   
3  B1 2019-07-27  19:00           Waregem     Mechelen     0     2   A   0.0   
4  B1 2019-07-27  19:30  Waasland-Beveren  Club Brugge     1     3   A   1.0   

   HTAG  ... Referee  Unnamed: 121  Unnamed: 106          kickoff_dt  season  \
0   1.0  ...     NaN           NaN           NaN 2019-07-26 19:30:00    2019   
1   0.0  ...     NaN           NaN           NaN 2019-07-27 17:00:00    2019   
2   1.0  ...     NaN           NaN           NaN 2019-07-27 19:00

In [ ]:
# ---------------------------
# 3) BASELINE-LIKE EXPORT (same characteristics as your baseline notebook)
#     Only pre-match identifiers & calendar info + target (no leakage)
# ---------------------------
baseline_cols = ['Div','Date','Time','HomeTeam','AwayTeam','country','league','season','month','weekday','is_weekend','y_over25']
baseline_like = df.reindex(columns=[c for c in baseline_cols if c in df.columns]).copy()
baseline_path = os.path.join(outdir, 'baseline_like.csv')
baseline_like.to_csv(baseline_path, index=False)
print(f"Saved baseline_like -> {baseline_path}  ({len(baseline_like):,} rows)")


Saved baseline_like -> /content/gdrive/MyDrive/ML2_SHARED/semestral_project/processed/baseline_like.csv  (42,593 rows)


In [ ]:
# ---------------------------
# 4) STATS-ONLY FEATURES (Variant i)
# ---------------------------
def _result_code(r):
    # +1 win, 0 draw, -1 loss from perspective of the row's team side
    return r

def compute_result_code(fthg, ftag, is_home: bool) -> int:
    if is_home:
        if fthg > ftag: return 1
        if fthg == ftag: return 0
        return -1
    else:
        if ftag > fthg: return 1
        if fthg == ftag: return 0
        return -1

def streak_series(results: pd.Series) -> pd.Series:
    # +N wins in a row / -N losses in a row (draw resets)
    streak = []
    cur = 0
    for r in results.fillna(0):
        if r == 0:
            cur = 0
        elif r > 0:
            cur = cur + 1 if cur >= 0 else 1
        else:
            cur = cur - 1 if cur <= 0 else -1
        streak.append(cur)
    return pd.Series(streak, index=results.index)

def build_team_rolling_features(df: pd.DataFrame, windows: List[int] = [5,10]) -> pd.DataFrame:
    d = df.copy()

    # Build per-team event view
    home = d[['Div','kickoff_dt','season','HomeTeam','AwayTeam','FTHG','FTAG','month','weekday','is_weekend']].copy()
    home['team'] = home['HomeTeam']; home['opp'] = home['AwayTeam']; home['is_home'] = 1
    home['gf'] = home['FTHG']; home['ga'] = home['FTAG']
    home['result'] = home.apply(lambda r: compute_result_code(r['FTHG'], r['FTAG'], True), axis=1)

    away = d[['Div','kickoff_dt','season','HomeTeam','AwayTeam','FTHG','FTAG','month','weekday','is_weekend']].copy()
    away['team'] = away['AwayTeam']; away['opp'] = away['HomeTeam']; away['is_home'] = 0
    away['gf'] = away['FTAG']; away['ga'] = away['FTHG']
    away['result'] = away.apply(lambda r: compute_result_code(r['FTHG'], r['FTAG'], False), axis=1)

    ev = pd.concat([home, away], ignore_index=True).sort_values(['Div','team','kickoff_dt']).reset_index(drop=True)
    ev['points'] = ev['result'].map({1:3, 0:1, -1:0})

    # Rest days
    ev['rest_days'] = ev.groupby(['Div','team'])['kickoff_dt'].diff().dt.total_seconds() / (3600*24)
    ev['rest_days'] = ev['rest_days'].fillna(ev['rest_days'].median())

    def add_rolls(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()
        g['streak'] = streak_series(g['result'])
        for W in windows:
            g[f'ppg_{W}'] = g['points'].shift(1).rolling(W, min_periods=1).mean()
            g[f'win_rate_{W}'] = (g['result'].shift(1) == 1).rolling(W, min_periods=1).mean()
            g[f'loss_rate_{W}'] = (g['result'].shift(1) == -1).rolling(W, min_periods=1).mean()
            g[f'gf_mean_{W}'] = g['gf'].shift(1).rolling(W, min_periods=1).mean()
            g[f'ga_mean_{W}'] = g['ga'].shift(1).rolling(W, min_periods=1).mean()
            g[f'gd_mean_{W}'] = (g['gf'] - g['ga']).shift(1).rolling(W, min_periods=1).mean()
            g[f'gd_std_{W}']  = (g['gf'] - g['ga']).shift(1).rolling(W, min_periods=2).std()
            g[f'total_goals_mean_{W}'] = (g['gf'] + g['ga']).shift(1).rolling(W, min_periods=1).mean()
            # contextual (home/away) win rates
            g[f'home_win_rate_{W}'] = np.where(g['is_home']==1,
                                               (g['result'].shift(1) == 1).rolling(W, min_periods=1).mean(),
                                               np.nan)
            g[f'away_win_rate_{W}'] = np.where(g['is_home']==0,
                                               (g['result'].shift(1) == 1).rolling(W, min_periods=1).mean(),
                                               np.nan)
                # --- WITH this index-aligned cumulative formulation ---
            g['_cnt']      = g.groupby('season').cumcount()                          # 0,1,2,...
            g['_ppg_cum']  = g.groupby('season')['points'].cumsum()
            g['_gf_cum']   = g.groupby('season')['gf'].cumsum()
            g['_ga_cum']   = g.groupby('season')['ga'].cumsum()

            # mean up to previous match in the same season (shift(1) excludes current row)
            denom = g['_cnt'].replace(0, np.nan)
            g['ppg_season'] = g['_ppg_cum'].shift(1) / denom
            g['gf_season']  = g['_gf_cum'].shift(1)  / denom
            g['ga_season']  = g['_ga_cum'].shift(1)  / denom

            # clean helpers & keep streak as "before current"
            g['streak'] = g['streak'].shift(1).fillna(0)
            g.drop(columns=['_cnt','_ppg_cum','_gf_cum','_ga_cum'], inplace=True)

            return g

    # keep grouping keys out of the function frame; avoid future pandas behavior changes
    ev = ev.groupby(['Div','team'], group_keys=False).apply(add_rolls)
    ev = ev.reset_index(drop=True)

    ev = ev.sort_values(['Div','season','kickoff_dt']).reset_index(drop=True)
    ev['total_goals_team'] = ev['gf'] + ev['ga']

    grp = ev.groupby(['Div','season'], sort=False)
    ev['_cnt_lg'] = grp.cumcount()                                  # 0,1,2,...
    ev['_cum_total'] = grp['total_goals_team'].cumsum()

    # league mean up to previous match (exclude current row)
    ev['league_avg_goals'] = ev['_cum_total'].shift(1) / ev['_cnt_lg'].replace(0, np.nan)

    # clean helpers
    ev.drop(columns=['total_goals_team','_cnt_lg','_cum_total'], inplace=True)

    # Pivot back to match-level
    home_feats = ev[ev['is_home']==1].copy()
    away_feats = ev[ev['is_home']==0].copy()

    roll_cols = [c for c in ev.columns if re.match(r'^(ppg_|win_rate_|loss_rate_|gf_mean_|ga_mean_|gd_mean_|gd_std_|total_goals_mean_|home_win_rate_|away_win_rate_|ppg_season|gf_season|ga_season|streak|rest_days|league_avg_goals)', c)]
    home_feats = home_feats[['Div','kickoff_dt','team'] + roll_cols] \
    .rename(columns=lambda c: f'home_{c}' if c not in ['Div','kickoff_dt','team'] else c)
    away_feats = away_feats[['Div','kickoff_dt','team'] + roll_cols] \
    .rename(columns=lambda c: f'away_{c}' if c not in ['Div','kickoff_dt','team'] else c)
    m = (d[['Div','kickoff_dt','HomeTeam','AwayTeam','season','month','weekday','is_weekend','y_over25']]
         .merge(home_feats, left_on=['Div','kickoff_dt','HomeTeam'], right_on=['Div','kickoff_dt','team'], how='left')
         .merge(away_feats, left_on=['Div','kickoff_dt','AwayTeam'], right_on=['Div','kickoff_dt','team'], how='left', suffixes=('','_away')))

    # symmetric diffs
    for c in roll_cols:
        hc, ac = f'home_{c}', f'away_{c}'
        if hc in m.columns and ac in m.columns:
            m[f'diff_{c}'] = m[hc] - m[ac]

    # drop helper 'team' columns
    for col in ['team','team_away']:
        if col in m.columns: m = m.drop(columns=[col])

    return m

stats_only = build_team_rolling_features(df, windows=[5,10])

# Optionally include identifiers for traceability
stats_only = stats_only.copy()
stats_only['country'] = df['country'].values
stats_only['league']  = df['league'].values

stats_path = os.path.join(outdir, 'features_stats_only.csv')
stats_only.to_csv(stats_path, index=False)
print(f"Saved stats-only -> {stats_path}  ({len(stats_only):,} rows)")

/tmp/ipython-input-3234785060.py:91: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ev = ev.groupby(['Div','team'], group_keys=False).apply(add_rolls)


Saved stats-only -> /content/gdrive/MyDrive/ML2_SHARED/semestral_project/processed/features_stats_only.csv  (42,593 rows)


In [ ]:
# ---------------------------
# 5) ODDS-AUGMENTED FEATURES (Variant ii)
#     Normalize odds -> implied probabilities; aggregate across bookmakers
# ---------------------------

# Known bookmaker prefixes (1X2 market) per notes
BOOKMAKER_PREFIXES_1X2 = [
    '1XB','B365','BF','BFD','BMGM','BV','BS','BWH','CL','GB','IW','LB','PS','SO','SB','SJ','SY','VC','WH'
]
# Map each prefix to expected suffix triplet (H,D,A)
TRIPLET = ('H','D','A')

def detect_1x2_triplets(columns: List[str]) -> Dict[str, Tuple[str,str,str]]:
    found = {}
    colset = set(columns)
    for pref in BOOKMAKER_PREFIXES_1X2:
        H, D, A = f"{pref}H", f"{pref}D", f"{pref}A"
        if H in colset and D in colset and A in colset:
            found[pref] = (H, D, A)
    # Also include market aggregates if present
    if {'MaxH','MaxD','MaxA'}.issubset(colset):
        found['Max'] = ('MaxH','MaxD','MaxA')
    if {'AvgH','AvgD','AvgA'}.issubset(colset):
        found['Avg'] = ('AvgH','AvgD','AvgA')
    if {'BbMxH','BbMxD','BbMxA'}.issubset(colset):
        found['BbMx'] = ('BbMxH','BbMxD','BbMxA')
    if {'BbAvH','BbAvD','BbAvA'}.issubset(colset):
        found['BbAv'] = ('BbAvH','BbAvD','BbAvA')
    return found

def implied_probs_1x2_row(row: pd.Series, cols: Tuple[str,str,str]) -> Tuple[float,float,float,float]:
    h,d,a = cols
    try:
        oh, od, oa = float(row[h]), float(row[d]), float(row[a])
        if oh <= 1 or od <= 1 or oa <= 1:
            return np.nan, np.nan, np.nan, np.nan
        inv_sum = (1/oh) + (1/od) + (1/oa)
        return (1/oh)/inv_sum, (1/od)/inv_sum, (1/oa)/inv_sum, inv_sum - 1.0
    except Exception:
        return np.nan, np.nan, np.nan, np.nan

# OU 2.5 detection (columns like 'B365>2.5', 'B365<2.5', 'Avg>2.5', etc.)
BOOKMAKER_PREFIXES_OU = ['B365','P','GB','Max','Avg','BbMx','BbAv']
def detect_ou_pairs(columns: List[str]) -> Dict[str, Tuple[str,str]]:
    found = {}
    colset = set(columns)
    for pref in BOOKMAKER_PREFIXES_OU:
        over, under = f"{pref}>2.5", f"{pref}<2.5"
        if over in colset and under in colset:
            found[pref] = (over, under)
    return found

def implied_probs_ou_row(row: pd.Series, cols: Tuple[str,str]) -> Tuple[float,float,float]:
    over_col, under_col = cols
    try:
        oo, uu = float(row[over_col]), float(row[under_col])
        if oo <= 1 or uu <= 1:
            return np.nan, np.nan, np.nan
        inv_sum = (1/oo) + (1/uu)
        return (1/oo)/inv_sum, (1/uu)/inv_sum, inv_sum - 1.0
    except Exception:
        return np.nan, np.nan, np.nan

def aggregate_market_probs(d: pd.DataFrame) -> pd.DataFrame:
    dfm = d.copy()
    cols = dfm.columns.tolist()

    # --- 1X2 aggregation
    triplets = detect_1x2_triplets(cols)
    pH_cols, pD_cols, pA_cols, or_cols = [], [], [], []
    for tag, trip in triplets.items():
        p = dfm.apply(implied_probs_1x2_row, cols=trip, axis=1, result_type='expand')
        dfm[f'pH_{tag}'], dfm[f'pD_{tag}'], dfm[f'pA_{tag}'], dfm[f'or1x2_{tag}'] = p[0], p[1], p[2], p[3]
        pH_cols.append(f'pH_{tag}'); pD_cols.append(f'pD_{tag}'); pA_cols.append(f'pA_{tag}'); or_cols.append(f'or1x2_{tag}')

    def _agg(cols_list: List[str], prefix: str) -> pd.DataFrame:
        if not cols_list:
            return pd.DataFrame(index=dfm.index)
        out = pd.DataFrame({
            f'{prefix}_mean': dfm[cols_list].mean(axis=1, skipna=True),
            f'{prefix}_std':  dfm[cols_list].std(axis=1, skipna=True),
            f'{prefix}_min':  dfm[cols_list].min(axis=1, skipna=True),
            f'{prefix}_max':  dfm[cols_list].max(axis=1, skipna=True),
            f'{prefix}_count': dfm[cols_list].notna().sum(axis=1)
        })
        return out

    agg_1x2 = pd.concat([
        _agg(pH_cols, 'mkt1x2_pH'),
        _agg(pD_cols, 'mkt1x2_pD'),
        _agg(pA_cols, 'mkt1x2_pA'),
        _agg(or_cols, 'mkt1x2_overround')
    ], axis=1)

    # --- OU 2.5 aggregation
    ou_pairs = detect_ou_pairs(cols)
    pOver_cols, pUnder_cols, or_ou_cols = [], [], []
    for tag, pair in ou_pairs.items():
        p = dfm.apply(implied_probs_ou_row, cols=pair, axis=1, result_type='expand')
        dfm[f'pOver25_{tag}'], dfm[f'pUnder25_{tag}'], dfm[f'orOU_{tag}'] = p[0], p[1], p[2]
        pOver_cols.append(f'pOver25_{tag}'); pUnder_cols.append(f'pUnder25_{tag}'); or_ou_cols.append(f'orOU_{tag}')

    agg_ou = pd.concat([
        _agg(pOver_cols,  'ou25_pOver'),
        _agg(pUnder_cols, 'ou25_pUnder'),
        _agg(or_ou_cols,  'ou25_overround')
    ], axis=1)

    # Simple differentials
    agg_extra = pd.DataFrame(index=dfm.index)
    if 'mkt1x2_pH_mean' in agg_1x2.columns and 'mkt1x2_pA_mean' in agg_1x2.columns:
        agg_extra['mkt_pH_minus_pA_mean'] = agg_1x2['mkt1x2_pH_mean'] - agg_1x2['mkt1x2_pA_mean']
    if 'ou25_pOver_mean' in agg_ou.columns and 'ou25_pUnder_mean' in agg_ou.columns:
        agg_extra['ou25_pOver_minus_pUnder_mean'] = agg_ou['ou25_pOver_mean'] - agg_ou['ou25_pUnder_mean']

    # Bookmaker counts (liquidity proxy)
    agg_extra['n_bookmakers_1x2'] = agg_1x2.filter(like='_count').iloc[:,0] if agg_1x2.filter(like='_count').shape[1] else 0
    agg_extra['n_bookmakers_ou']  = agg_ou.filter(like='_count').iloc[:,0] if agg_ou.filter(like='_count').shape[1] else 0

    # Asian handicap — optional (line if available)
    for col in ['AHh','BbAHh','B365AH']:
        if col in dfm.columns:
            agg_extra['ah_line'] = dfm[col]
            break

    # Concatenate
    market_agg = pd.concat([agg_1x2, agg_ou, agg_extra], axis=1)
    market_agg.index = dfm.index
    return market_agg

market_feats = aggregate_market_probs(df)

# Use a unique match key to avoid cross-joins: Div + kickoff_dt + teams
key_cols = ['Div', 'kickoff_dt', 'HomeTeam', 'AwayTeam']

# Build market_view with the same keys
market_view = pd.concat([df[key_cols], market_feats], axis=1)

# Merge market aggregates onto the stats-only frame
# If each match in df is unique by key_cols, you can validate='many_to_one' or 'one_to_one'
odds_aug = stats_only.merge(market_view, on=key_cols, how='left', validate='many_to_one')

# country/league already exist in stats_only (from earlier step) — no reassignment needed
# If they don't, do a safe merge instead of direct assignment:
# id_cols = key_cols + ['country','league']
# odds_aug = odds_aug.merge(df[id_cols].drop_duplicates(key_cols), on=key_cols, how='left', validate='many_to_one')

odds_path = os.path.join(outdir, 'features_odds_aug.csv')
odds_aug.to_csv(odds_path, index=False)
print(f"Saved odds-augmented -> {odds_path}  ({len(odds_aug):,} rows)")


Saved odds-augmented -> /content/gdrive/MyDrive/ML2_SHARED/semestral_project/processed/features_odds_aug.csv  (42,593 rows)


In [ ]:
odds_aug.head()

,Div,kickoff_dt,HomeTeam,AwayTeam,season,month,weekday,is_weekend,y_over25,home_rest_days,...,ou25_overround_mean,ou25_overround_std,ou25_overround_min,ou25_overround_max,ou25_overround_count,mkt_pH_minus_pA_mean,ou25_pOver_minus_pUnder_mean,n_bookmakers_1x2,n_bookmakers_ou,ah_line
0,B1,2019-07-26 19:30:00,Genk,Kortrijk,2019,7,4,0,1,7.0,...,0.049093,0.022381,0.023949,0.070261,4,0.565998,0.248464,7,4,-1.50
1,B1,2019-07-27 17:00:00,Cercle Brugge,Standard,2019,7,5,1,0,7.0,...,0.042489,0.023342,0.010280,0.060150,4,-0.243265,0.078263,7,4,0.50
2,B1,2019-07-27 19:00:00,St Truiden,Mouscron,2019,7,5,1,0,7.0,...,0.040507,0.019018,0.015007,0.055556,4,0.240384,0.042186,7,4,-0.50
3,B1,2019-07-27 19:00:00,Waregem,Mechelen,2019,7,5,1,0,7.0,...,0.050337,0.018137,0.028397,0.067526,4,0.119699,0.145606,7,4,-0.25
4,B1,2019-07-27 19:30:00,Waasland-Beveren,Club Brugge,2019,7,5,1,1,7.0,...,0.051469,0.015247,0.032911,0.065329,4,-0.455469,0.217558,7,4,1.00


In [ ]:
# --- quick column diagnostics for feature selection ---

import pandas as pd

def summarize_feature_columns(df_stats, df_odds):
    baseline_like = ['Div','Date','Time','HomeTeam','AwayTeam','season','month','weekday','is_weekend','y_over25']

    new_stats_cols = sorted(set(df_stats.columns) - set(baseline_like))
    new_odds_cols  = sorted(set(df_odds.columns) - set(df_stats.columns))

    print(f"🧮 Stats-only total cols: {len(df_stats.columns)}")
    print(f"→ newly generated stats features: {len(new_stats_cols)}")
    print(f"🧮 Odds-augmented total cols: {len(df_odds.columns)}")
    print(f"→ additional market features: {len(new_odds_cols)}\n")

    # group by prefix
    groups = {
        'home_feats'  : [c for c in new_stats_cols if c.startswith('home_')],
        'away_feats'  : [c for c in new_stats_cols if c.startswith('away_')],
        'diff_feats'  : [c for c in new_stats_cols if c.startswith('diff_')],
        'league_feats': [c for c in new_stats_cols if 'league_avg' in c],
        'market_feats': [c for c in new_odds_cols if c.startswith(('mkt','ou','ah'))],
    }

    for name, cols in groups.items():
        print(f"▸ {name}: {len(cols)} columns")
        if cols:
            print(", ".join(cols[:8]) + ("..." if len(cols) > 8 else ""))
        print()

    return groups

groups = summarize_feature_columns(stats_only, odds_aug)

# optional: peek at some feature stats for later filtering
def preview_features(df, cols, n=5):
    print(f"\nPreview ({len(cols)} cols): showing {min(n, len(cols))}")
    display(df[cols[:n]].describe(percentiles=[.1,.5,.9]).T.round(3))

# Example: look at a few 'diff' features and 'market' features
preview_features(stats_only, groups['diff_feats'])
preview_features(odds_aug, groups['market_feats'])

🧮 Stats-only total cols: 59
→ newly generated stats features: 51
🧮 Odds-augmented total cols: 99
→ additional market features: 40

▸ home_feats: 16 columns
home_away_win_rate_5, home_ga_mean_5, home_ga_season, home_gd_mean_5, home_gd_std_5, home_gf_mean_5, home_gf_season, home_home_win_rate_5...

▸ away_feats: 16 columns
away_away_win_rate_5, away_ga_mean_5, away_ga_season, away_gd_mean_5, away_gd_std_5, away_gf_mean_5, away_gf_season, away_home_win_rate_5...

▸ diff_feats: 16 columns
diff_away_win_rate_5, diff_ga_mean_5, diff_ga_season, diff_gd_mean_5, diff_gd_std_5, diff_gf_mean_5, diff_gf_season, diff_home_win_rate_5...

▸ league_feats: 3 columns
away_league_avg_goals, diff_league_avg_goals, home_league_avg_goals

▸ market_feats: 38 columns
ah_line, mkt1x2_overround_count, mkt1x2_overround_max, mkt1x2_overround_mean, mkt1x2_overround_min, mkt1x2_overround_std, mkt1x2_pA_count, mkt1x2_pA_max...


Preview (16 cols): showing 5


,count,mean,std,min,10%,50%,90%,max
diff_away_win_rate_5,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
diff_ga_mean_5,42161.0,0.025,0.823,-5.000,-1.000,0.0,1.000,7.000
diff_ga_season,41404.0,0.015,0.658,-5.000,-0.733,0.0,0.750,7.000
diff_gd_mean_5,42161.0,-0.051,1.313,-8.000,-1.600,0.0,1.600,8.000
diff_gd_std_5,41730.0,0.001,0.884,-4.458,-1.113,0.0,1.096,4.356



Preview (38 cols): showing 5


,count,mean,std,min,10%,50%,90%,max
ah_line,42527.0,-0.263,0.690,-3.750,-1.000,-0.250,0.500,3.750
mkt1x2_overround_count,42593.0,6.819,0.463,0.000,6.000,7.000,7.000,7.000
mkt1x2_overround_max,42547.0,0.082,0.018,0.044,0.062,0.078,0.107,0.310
mkt1x2_overround_mean,42547.0,0.053,0.010,0.015,0.041,0.053,0.064,0.098
mkt1x2_overround_min,42547.0,0.012,0.012,-0.129,-0.003,0.012,0.028,0.071


In [ ]:
# --- show all generated columns grouped logically ---
pd.set_option("display.max_colwidth", None)

for name, cols in groups.items():
    print(f"\n🔹 {name.upper()} ({len(cols)} columns):\n")
    for c in cols:
        print(c)


🔹 HOME_FEATS (16 columns):

home_away_win_rate_5
home_ga_mean_5
home_ga_season
home_gd_mean_5
home_gd_std_5
home_gf_mean_5
home_gf_season
home_home_win_rate_5
home_league_avg_goals
home_loss_rate_5
home_ppg_5
home_ppg_season
home_rest_days
home_streak
home_total_goals_mean_5
home_win_rate_5

🔹 AWAY_FEATS (16 columns):

away_away_win_rate_5
away_ga_mean_5
away_ga_season
away_gd_mean_5
away_gd_std_5
away_gf_mean_5
away_gf_season
away_home_win_rate_5
away_league_avg_goals
away_loss_rate_5
away_ppg_5
away_ppg_season
away_rest_days
away_streak
away_total_goals_mean_5
away_win_rate_5

🔹 DIFF_FEATS (16 columns):

diff_away_win_rate_5
diff_ga_mean_5
diff_ga_season
diff_gd_mean_5
diff_gd_std_5
diff_gf_mean_5
diff_gf_season
diff_home_win_rate_5
diff_league_avg_goals
diff_loss_rate_5
diff_ppg_5
diff_ppg_season
diff_rest_days
diff_streak
diff_total_goals_mean_5
diff_win_rate_5

🔹 LEAGUE_FEATS (3 columns):

away_league_avg_goals
diff_league_avg_goals
home_league_avg_goals

🔹 MARKET_FEATS (38 colum